# Authors FCN checkpoints ? Test_Modelsv0v0
Runs the authors' model, preprocessing, four 256x256 tiles, and original metric protocol. Set `MODEL_ID`/`DATASET_VERSION` below or use the batch runner. Default is the original v1.0 test data; v1.1 is a separate cross-version comparison.

Only local I/O, device compatibility, metric export, and prediction/visualization exports are adapted. Historical label conversion is preserved, including its behavior on tiles without water. Independently scored integer-label metrics in the report are kept separate. No training is performed.

In [ ]:
from pathlib import Path
import os, sys, json, hashlib
from datetime import datetime
from uuid import uuid4
for repository_dir in (Path("."), Path(".."), Path("../.."), Path("../../..")):
    if (repository_dir / "src/use_croma.py").is_file():
        os.chdir(repository_dir)
        break
else:
    raise FileNotFoundError("Open this notebook from within the repository.")
sys.path.insert(0, "src")
MODEL_ID = os.environ.get("FCNN_MODEL", "hand_labeled")
DATASET_VERSION = os.environ.get("FCNN_DATASET", "v1.0")
assert MODEL_ID in {"hand_labeled", "s1_weak", "s2_weak", "permanent_water"}
assert DATASET_VERSION in {"v1.0", "v1.1"}
TRAIN_VERSION = "v1.0"
DATA_ROOT = "../Sen1Floods11_experiment/datasets" if DATASET_VERSION == "v1.0" else "dataset/sen1floods11_hand_labeled_only/HandLabeled"
SPLIT_DIR = DATA_ROOT if DATASET_VERSION == "v1.0" else "dataset/sen1floods11_hand_labeled_only/flood_handlabeled"
EXPERIMENTS = {
 "hand_labeled": {"display_name": "Authors Hand Labeled", "checkpoint": "5e4_handlabeled_0_8_0.43163052201271057.cp"},
 "s1_weak": {"display_name": "Authors S1 Weak", "checkpoint": "5e4_s1weak_2_3_0.45159465074539185.cp"},
 "s2_weak": {"display_name": "Authors S2 Weak", "checkpoint": "5e4_flood_0_89_0.5418742895126343.cp"},
 "permanent_water": {"display_name": "Authors Permanent Water", "checkpoint": "5e4_permanent_0_180_0.8432363867759705.cp"},
}
EXPERIMENT_NAME = MODEL_ID
MODEL_NAME = EXPERIMENTS[MODEL_ID]["display_name"]
LOCAL_CHECKPOINT = str(Path("pretrained/sen1floods11_authors") / EXPERIMENTS[MODEL_ID]["checkpoint"])
CHECKPOINT_PATH = LOCAL_CHECKPOINT
run_id = os.environ.get("FCNN_RUN_ID") or (datetime.now().astimezone().strftime("%Y-%m-%d_%H-%M-%S_%f") + "_" + uuid4().hex[:8])
LOCAL_RESULTS = Path("runs/sen1floods11_reproduction") / ("authors_" + MODEL_ID) / f"test_{Path(CHECKPOINT_PATH).stem}_{DATASET_VERSION}_{run_id}"
LOCAL_RESULTS.mkdir(parents=True, exist_ok=False)
with (LOCAL_RESULTS / "run.json").open("w") as f:
 json.dump({"model":MODEL_NAME,"phase":"test","test_dataset":DATASET_VERSION,"training_dataset":"v1.0","checkpoint":CHECKPOINT_PATH,"checkpoint_sha256":hashlib.sha256(Path(CHECKPOINT_PATH).read_bytes()).hexdigest(),"started_at":datetime.now().astimezone().isoformat(),"notebook":"experiments/sen1floods11_reproduction/authors/Test_Modelsv0v0.ipynb","protocol":"Authors processTestIm and category/metric functions; legacy label conversion preserved"},f,indent=2)
print("Run directory:", LOCAL_RESULTS)
def resolve_dataset_path(value):
    if DATASET_VERSION == "v1.0":
        return str(Path(DATA_ROOT) / value)
    name = Path(value).name
    folder = name.rsplit("_", 1)[1].removesuffix(".tif")
    return str(Path(DATA_ROOT) / folder / name)


In [ ]:
from time import time
import csv
from PIL import Image
import rasterio
import numpy as np
import os
BASEDIR = DATA_ROOT + '/' 

def getArrFlood(fname):
  return rasterio.open(fname).read()

def download_flood_water_data_from_list(l):
  i = 0
  tot_nan = 0
  tot_good = 0
  flood_data = []
  for (im_fname, mask_fname) in l:
    if not os.path.exists(im_fname):
      raise FileNotFoundError(im_fname)
    arr_x = np.nan_to_num(getArrFlood(im_fname))
    arr_y = np.uint8(getArrFlood(mask_fname))

    # Safe handling of ignore labels (-1)
    ignore = (arr_y == 255) | (arr_y == 254)  # equivalent to -1 in signed/unsigned uint8 overflows
    arr_y = np.uint8(getArrFlood(mask_fname))

    if np.sum((arr_y != arr_y)) == 0:
      arr_x = np.clip(arr_x, -50, 1)
      arr_x = (arr_x + 50) / 51
      if i % 100 == 0:
        print(i)
        print(im_fname, mask_fname)
      i += 1
      flood_data.append((arr_x, arr_y))
    else:
      print("skipping nan")
  print(i)
  return flood_data

def download_flood_water_classification_data_from_list(l):
  i = 0
  tot_nan = 0
  tot_good = 0
  flood_data = []
  for (im_fname, mask_fname) in l:
    if not os.path.exists(im_fname):
      print(im_fname)
      continue
    arr_x = np.nan_to_num(getArrFlood(im_fname))
    arr_y = np.uint8(getArrFlood(mask_fname))

    if np.sum((arr_y != arr_y)) == 0:
      if i % 100 == 0:
        print(i)
        print(im_fname, mask_fname)
      i += 1
      flood_data.append((arr_x, arr_y))
    else:
      print("skipping nan")
  print(i)
  return flood_data


def load_flood_train_data():
  fname = os.path.join(SPLIT_DIR, "flood_train_data.csv")
  with open(fname) as f:
    fname = [tuple(resolve_dataset_path(v) for v in line) for line in csv.reader(f)]
  return download_flood_water_data_from_list(fname)

def load_weak_flood_train_data():
  files = [(os.path.join("S1_NoQC", x[1]), os.path.join("NoQC", x[0])) for x in zip(sorted(os.listdir("files4/NoQC")), sorted(os.listdir("files4/S1_NoQC")))]
  files = [x for x in files if "Bolivia" not in x[0]]
  print(files[0:10])
  return download_flood_water_data_from_list(files)

def load_weak_flood_S1_train_data():
  files = [(os.path.join("S1_NoQC", x[1]), os.path.join("S1Flood_NoQC", x[0])) for x in zip(sorted(os.listdir("files4/S1Flood_NoQC")), sorted(os.listdir("files4/S1_NoQC")))]
  print(files[0:10])
  return download_flood_water_data_from_list(files)


def load_flood_valid_data():
  fname = os.path.join(SPLIT_DIR, "flood_valid_data.csv")
  with open(fname) as f:
    fname = [tuple(resolve_dataset_path(v) for v in line) for line in csv.reader(f)]
  print(fname, "files!")
  return download_flood_water_data_from_list(fname)

def load_flood_test_data():
  fname = os.path.join(SPLIT_DIR, "flood_test_data.csv")
  with open(fname) as f:
    fname = [tuple(resolve_dataset_path(v) for v in line) for line in csv.reader(f)]
  return download_flood_water_data_from_list(fname)

def load_flood_test_perm_data():
  fname = os.path.join(SPLIT_DIR, "flood_test_data.csv")
  with open(fname) as f:
    fname = [tuple(resolve_dataset_path(v) for v in line) for line in csv.reader(f)]
  fname = [(t[0], t[1].replace("QC_v2", "Perm").replace("QC", "Perm").replace("LabelHand", "JRCWaterHand")) for t in fname]
  return download_flood_water_data_from_list(fname)

def load_flood_bolivia_test_data():
  fname = os.path.join(SPLIT_DIR, "flood_bolivia_data.csv")
  with open(fname) as f:
    fname = [tuple(resolve_dataset_path(v) for v in line) for line in csv.reader(f)]
  return download_flood_water_data_from_list(fname)

def load_flood_bolivia_test_perm_data():
  fname = os.path.join(SPLIT_DIR, "flood_bolivia_data.csv")
  with open(fname) as f:
    fname = [tuple(resolve_dataset_path(v) for v in line) for line in csv.reader(f)]
  fname = [(t[0], t[1].replace("QC_v2", "Perm").replace("QC", "Perm").replace("LabelHand", "JRCWaterHand")) for t in fname]
  return download_flood_water_data_from_list(fname)

def load_flood_test_trad_data():
  fname = os.path.join(SPLIT_DIR, "flood_test_data.csv")
  with open(fname) as f:
    fname = [tuple(resolve_dataset_path(v) for v in line) for line in csv.reader(f)]
  fname = [getTradFname(x) for x in fname]
  return download_flood_water_classification_data_from_list(fname)

def load_flood_bolivia_test_trad_data():
  fname = os.path.join(SPLIT_DIR, "flood_bolivia_data.csv")
  with open(fname) as f:
    fname = [tuple(resolve_dataset_path(v) for v in line) for line in csv.reader(f)]
  fname = [getTradFname(x) for x in fname]
  return download_flood_water_classification_data_from_list(fname)

def getTradFname(a):
  x,y = a
  x = x.replace("S1", "S1Flood")
  return x,y

In [ ]:
flood_test_perm_data = load_flood_test_perm_data()
flood_test_all_data = load_flood_test_data()

flood_test_bolivia_perm_data = load_flood_bolivia_test_perm_data()
flood_test_bolivia_all_data = load_flood_bolivia_test_data()

In [ ]:
def ignore_perm_water(all_water, perm_water):
  w = all_water.copy()
  perm_water = (w == 1) * (perm_water == 1)
  w[perm_water] = 255
  return w

def ignore_flood_water(all_water, perm_water):
  w = all_water.copy()
  flood_water = (w == 1) * (perm_water == 0)
  w[flood_water] = 255
  return w

def get_flood_flood_test_data(all_water_data, perm_water_data):
  flood_flood_test_data = []
  for ((awx, awy), (pwx, pwy)) in zip(all_water_data, perm_water_data):
    pwy = ignore_perm_water(awy, pwy)
    if np.sum(awy == 1) > 0 and np.sum(pwy == 1) == 0:
      continue
    flood_flood_test_data.append((awx, pwy))
  return flood_flood_test_data


def get_perm_flood_test_data(all_water_data, perm_water_data):
  perm_flood_test_data = []
  for ((awx, awy), (pwx, pwy)) in zip(all_water_data, perm_water_data):
    pwy = ignore_flood_water(awy, pwy)
    if np.sum(awy == 1) > 0 and np.sum(pwy == 1) == 0:
      continue
    perm_flood_test_data.append((awx, pwy))
  return perm_flood_test_data


In [ ]:
flood_flood_test_data = get_flood_flood_test_data(flood_test_all_data, flood_test_perm_data)
perm_flood_test_data = get_perm_flood_test_data(flood_test_all_data, flood_test_perm_data)

flood_flood_bolivia_test_data = get_flood_flood_test_data(flood_test_bolivia_all_data, flood_test_bolivia_perm_data)
perm_flood_bolivia_test_data = get_perm_flood_test_data(flood_test_bolivia_all_data, flood_test_bolivia_perm_data)

In [ ]:
import torch
from torchvision import transforms
import torchvision.transforms.functional as F
import random

class InMemoryDataset(torch.utils.data.Dataset):

  def __init__(self, data_list, preprocess_func):
    self.data_list = data_list
    self.preprocess_func = preprocess_func

  def __getitem__(self, i):
    return self.preprocess_func(self.data_list[i])

  def __len__(self):
    return len(self.data_list)


def processAndAugment(data):
  (x,y) = data
  im,label = x.copy(), y.copy()

  # convert to PIL for easier transforms
  im1 = Image.fromarray(im[0])
  im2 = Image.fromarray(im[1])
  label = Image.fromarray(label.squeeze())

  # Get params for random transforms
  i, j, h, w = transforms.RandomCrop.get_params(im1, (256, 256))

  im1 = F.crop(im1, i, j, h, w)
  im2 = F.crop(im2, i, j, h, w)
  label = F.crop(label, i, j, h, w)
  if random.random() > 0.5:
    im1 = F.hflip(im1)
    im2 = F.hflip(im2)
    label = F.hflip(label)
  if random.random() > 0.5:
    im1 = F.vflip(im1)
    im2 = F.vflip(im2)
    label = F.vflip(label)

  norm = transforms.Normalize([0.6851, 0.5235], [0.0820, 0.1102])
  im = torch.stack([transforms.ToTensor()(im1).squeeze(), transforms.ToTensor()(im2).squeeze()])
  im = norm(im)
  label = transforms.ToTensor()(label).squeeze()
  if torch.sum(label.gt(.003) * label.lt(.004)):
    label *= 255
  label = label.round()

  return im, label


def processTestIm(data):
  (x,y) = data
  im,label = x.copy(), y.copy()
  norm = transforms.Normalize([0.6851, 0.5235], [0.0820, 0.1102])
  #label[0][0][0] = 255
  # convert to PIL for easier transforms
  im_c1 = Image.fromarray(im[0]).resize((512,512))
  im_c2 = Image.fromarray(im[1]).resize((512,512))
  label = Image.fromarray(label.squeeze()).resize((512,512))

  im_c1s = [F.crop(im_c1, 0, 0, 256, 256), F.crop(im_c1, 0, 256, 256, 256),
            F.crop(im_c1, 256, 0, 256, 256), F.crop(im_c1, 256, 256, 256, 256)]
  im_c2s = [F.crop(im_c2, 0, 0, 256, 256), F.crop(im_c2, 0, 256, 256, 256),
            F.crop(im_c2, 256, 0, 256, 256), F.crop(im_c2, 256, 256, 256, 256)]
  labels = [F.crop(label, 0, 0, 256, 256), F.crop(label, 0, 256, 256, 256),
            F.crop(label, 256, 0, 256, 256), F.crop(label, 256, 256, 256, 256)]


  ims = [torch.stack((transforms.ToTensor()(x).squeeze(),
                    transforms.ToTensor()(y).squeeze()))
                    for (x,y) in zip(im_c1s, im_c2s)]
  ims = [norm(im) for im in ims]
  ims = torch.stack(ims)
  labels = [(transforms.ToTensor()(label).squeeze()) for label in labels]
  labels = torch.stack(labels)
  if torch.sum(labels.gt(.003) * labels.lt(.004)):
    labels *= 255
  labels = labels.round()
  return ims, labels


In [ ]:
test_all_data = flood_test_all_data
test_all_dataset = InMemoryDataset(test_all_data, processTestIm)
test_all_loader = torch.utils.data.DataLoader(test_all_dataset, batch_size=1, shuffle=True, sampler=None,
                  batch_sampler=None, num_workers=0, collate_fn=lambda x: x[0],
                  pin_memory=True, drop_last=False, timeout=0,
                  worker_init_fn=None)
test_all_iter = iter(test_all_loader)

test_flood_data = flood_flood_test_data
test_flood_dataset = InMemoryDataset(test_flood_data, processTestIm)
test_flood_loader = torch.utils.data.DataLoader(test_flood_dataset, batch_size=1, shuffle=True, sampler=None,
                  batch_sampler=None, num_workers=0, collate_fn=lambda x: x[0],
                  pin_memory=True, drop_last=False, timeout=0,
                  worker_init_fn=None)
test_flood_iter = iter(test_flood_loader)

test_perm_data = perm_flood_test_data
test_perm_dataset = InMemoryDataset(test_perm_data, processTestIm)
test_perm_loader = torch.utils.data.DataLoader(test_perm_dataset, batch_size=1, shuffle=True, sampler=None,
                  batch_sampler=None, num_workers=0, collate_fn=lambda x: x[0],
                  pin_memory=True, drop_last=False, timeout=0,
                  worker_init_fn=None)
test_perm_iter = iter(test_perm_loader)

In [ ]:
test_bolivia_all_data = flood_test_bolivia_all_data
test_bolivia_all_dataset = InMemoryDataset(test_bolivia_all_data, processTestIm)
test_bolivia_all_loader = torch.utils.data.DataLoader(test_bolivia_all_dataset, batch_size=1, shuffle=True, sampler=None,
                  batch_sampler=None, num_workers=0, collate_fn=lambda x: x[0],
                  pin_memory=True, drop_last=False, timeout=0,
                  worker_init_fn=None)
test_bolivia_all_iter = iter(test_bolivia_all_loader)

test_bolivia_flood_data = flood_flood_bolivia_test_data
test_bolivia_flood_dataset = InMemoryDataset(test_bolivia_flood_data, processTestIm)
test_bolivia_flood_loader = torch.utils.data.DataLoader(test_bolivia_flood_dataset, batch_size=1, shuffle=True, sampler=None,
                  batch_sampler=None, num_workers=0, collate_fn=lambda x: x[0],
                  pin_memory=True, drop_last=False, timeout=0,
                  worker_init_fn=None)
test_bolivia_flood_iter = iter(test_flood_loader)

test_bolivia_perm_data = perm_flood_bolivia_test_data
test_bolivia_perm_dataset = InMemoryDataset(test_bolivia_perm_data, processTestIm)
test_bolivia_perm_loader = torch.utils.data.DataLoader(test_bolivia_perm_dataset, batch_size=1, shuffle=True, sampler=None,
                  batch_sampler=None, num_workers=0, collate_fn=lambda x: x[0],
                  pin_memory=True, drop_last=False, timeout=0,
                  worker_init_fn=None)
test_bolivia_perm_iter = iter(test_bolivia_perm_loader)

In [ ]:
import torchvision.models as models
import torch.nn as nn
import torch

net = models.segmentation.fcn_resnet50(pretrained=False, num_classes=2, pretrained_backbone=False)
net.backbone.conv1 = nn.Conv2d(2, 64, kernel_size=7, stride=2, padding=3,
                               bias=False)
def convertBNtoGN(module, num_groups=16):
  if isinstance(module, torch.nn.modules.batchnorm.BatchNorm2d):
    return nn.GroupNorm(num_groups, module.num_features,
                        eps=module.eps, affine=module.affine)
    if module.affine:
        mod.weight.data = module.weight.data.clone().detach()
        mod.bias.data = module.bias.data.clone().detach()

  for name, child in module.named_children():
      module.add_module(name, convertBNtoGN(child, num_groups=num_groups))

  return module

net = convertBNtoGN(net)
net

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
state_dict = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=True)
net.load_state_dict(state_dict.get("model_state_dict", state_dict))
net = net.to(device).eval()
print(MODEL_NAME, DATASET_VERSION, device)


In [ ]:
from tqdm.auto import tqdm
from IPython.display import clear_output

import os
import json
from datetime import datetime, timezone

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def computeIOU(output, target):
    output = torch.argmax(output, dim=1).flatten()
    target = target.flatten()

    no_ignore = target.ne(255).to(device)
    output = output.masked_select(no_ignore)
    target = target.masked_select(no_ignore)

    intersection = torch.sum(output * target)
    union = torch.sum(target) + torch.sum(output) - intersection

    iou = (intersection + 0.0000001) / (union + 0.0000001)

    if iou != iou:
        print("failed, replacing with 0")
        iou = torch.tensor(0).float()

    return iou


def computeAccuracy(output, target):
    output = torch.argmax(output, dim=1).flatten()
    target = target.flatten()

    no_ignore = target.ne(255).to(device)
    output = output.masked_select(no_ignore)
    target = target.masked_select(no_ignore)

    correct = torch.sum(output.eq(target))

    return correct.float() / len(target)


def truePositives(output, target):
    output = torch.argmax(output, dim=1).flatten()
    target = target.flatten()

    no_ignore = target.ne(255).to(device)
    output = output.masked_select(no_ignore)
    target = target.masked_select(no_ignore)

    correct = torch.sum(output * target)

    return correct


def trueNegatives(output, target):
    output = torch.argmax(output, dim=1).flatten()
    target = target.flatten()

    no_ignore = target.ne(255).to(device)
    output = output.masked_select(no_ignore)
    target = target.masked_select(no_ignore)

    output = (output == 0)
    target = (target == 0)

    correct = torch.sum(output * target)

    return correct


def falsePositives(output, target):
    output = torch.argmax(output, dim=1).flatten()
    target = target.flatten()

    no_ignore = target.ne(255).to(device)
    output = output.masked_select(no_ignore)
    target = target.masked_select(no_ignore)

    output = (output == 1)
    target = (target == 0)

    correct = torch.sum(output * target)

    return correct


def falseNegatives(output, target):
    output = torch.argmax(output, dim=1).flatten()
    target = target.flatten()

    no_ignore = target.ne(255).to(device)
    output = output.masked_select(no_ignore)
    target = target.masked_select(no_ignore)

    output = (output == 0)
    target = (target == 1)

    correct = torch.sum(output * target)

    return correct


def test_loop(test_data_loader, net):
    net = net.eval()
    net = net.to(device)

    count = 0
    iou = 0
    accuracy = 0

    tp = 0
    fp = 0
    tn = 0
    fn = 0

    with torch.no_grad():
        for images, labels in tqdm(test_data_loader):

            images = images.to(device)
            labels = labels.to(device)

            outputs = net(images)

            valid_iou = computeIOU(outputs["out"], labels)

            iou += valid_iou
            accuracy += computeAccuracy(outputs["out"], labels)

            tp += truePositives(outputs["out"], labels)
            fp += falsePositives(outputs["out"], labels)
            tn += trueNegatives(outputs["out"], labels)
            fn += falseNegatives(outputs["out"], labels)

            count += 1

    mean_iou = iou / count
    total_iou = tp.float() / (fn + fp + tp)
    omission = fn.float() / (fn + tp)
    commission = fp.float() / (tn + fp)
    mean_accuracy = accuracy / count

    print("Test Mean IOU:", mean_iou)
    print("Total IOU:", total_iou)
    print("OMISSION:", omission)
    print("COMMISSION:", commission)
    print("Test Accuracy:", mean_accuracy)

    return {
        "mean_iou": float(mean_iou.detach().cpu()),
        "total_iou": float(total_iou.detach().cpu()),
        "omission": float(omission.detach().cpu()),
        "commission": float(commission.detach().cpu()),
        "accuracy": float(mean_accuracy.detach().cpu()),
        "tp": int(tp.detach().cpu()),
        "fp": int(fp.detach().cpu()),
        "tn": int(tn.detach().cpu()),
        "fn": int(fn.detach().cpu()),
        "num_samples": int(count)
    }


# ============================================================
# LOAD SELECTED CHECKPOINT
# ============================================================

print("========================================")
print("EXPERIMENT:", MODEL_NAME)
print("CHECKPOINT:", CHECKPOINT_PATH)
print("DEVICE:", device)
print("========================================")

state_dict = torch.load(
    CHECKPOINT_PATH,
    map_location=device
)

net.load_state_dict(state_dict.get("model_state_dict", state_dict))
net = net.to(device)


# ============================================================
# RUN EVALUATION
# ============================================================

results = {}


print("\n========================================")
print("NON-BOLIVIA - ALL WATER")
print("========================================")

results["non_bolivia_all_water"] = test_loop(
    test_all_loader,
    net
)


print("\n========================================")
print("NON-BOLIVIA - FLOOD WATER")
print("========================================")

results["non_bolivia_flood_water"] = test_loop(
    test_flood_loader,
    net
)


print("\n========================================")
print("NON-BOLIVIA - PERMANENT WATER")
print("========================================")

results["non_bolivia_permanent_water"] = test_loop(
    test_perm_loader,
    net
)


print("\n\n========================================")
print("BOLIVIA")
print("========================================")


print("\n========================================")
print("BOLIVIA - ALL WATER")
print("========================================")

results["bolivia_all_water"] = test_loop(
    test_bolivia_all_loader,
    net
)


print("\n========================================")
print("BOLIVIA - FLOOD WATER")
print("========================================")

results["bolivia_flood_water"] = test_loop(
    test_bolivia_flood_loader,
    net
)


print("\n========================================")
print("BOLIVIA - PERMANENT WATER")
print("========================================")

results["bolivia_permanent_water"] = test_loop(
    test_bolivia_perm_loader,
    net
)


# ============================================================
# BUILD RESULT FILE
# ============================================================

timestamp = datetime.now(timezone.utc)

output = {
    "experiment_name": EXPERIMENT_NAME,

    "model": {
        "name": MODEL_NAME,
        "source": "mine" if MODEL_ID.startswith("mine") else "authors",
        "training_dataset_version": TRAIN_VERSION,
        "architecture": "FCN-ResNet50",
        "input_modality": "Sentinel-1 VV/VH",
        "checkpoint": os.path.basename(CHECKPOINT_PATH),
        "checkpoint_path": CHECKPOINT_PATH
    },

    "dataset": {
        "name": "Sen1Floods11",
        "version": DATASET_VERSION,
        "evaluation_set": "original hand-labeled test set"
    },

    "runtime": {
        "device": str(device),
        "cuda_available": torch.cuda.is_available(),
        "gpu": (
            torch.cuda.get_device_name(0)
            if torch.cuda.is_available()
            else None
        ),
        "torch_version": torch.__version__,
        "numpy_version": np.__version__
    },

    "timestamp_utc": timestamp.isoformat(),

    "results": results
}


# ============================================================
# SAVE JSON TO GOOGLE DRIVE
# ============================================================

RESULTS_DIR = str(LOCAL_RESULTS)

os.makedirs(
    RESULTS_DIR,
    exist_ok=True
)

filename = (
    f"{timestamp.strftime('%Y%m%dT%H%M%S')}_"
    f"{EXPERIMENT_NAME}.json"
)

output_path = os.path.join(
    RESULTS_DIR,
    filename
)

with open(output_path, "w") as f:
    json.dump(
        output,
        f,
        indent=2
    )


print("\n\n========================================")
print("EVALUATION COMPLETE")
print("========================================")
print("Experiment:", MODEL_NAME)
print("Checkpoint:", os.path.basename(CHECKPOINT_PATH))
print("Saved to:", output_path)

In [ ]:
# Additional export only: use the original processTestIm and net unchanged.
# Iterate the CSV-ordered in-memory arrays, not the shuffled metric loaders.
net.eval()
with torch.no_grad():
    for split, csv_name, samples in [
        ("non_bol", "flood_test_data.csv", flood_test_all_data),
        ("bol", "flood_bolivia_data.csv", flood_test_bolivia_all_data),
    ]:
        with open(os.path.join(SPLIT_DIR, csv_name)) as f:
            pairs = [tuple(resolve_dataset_path(v) for v in row) for row in csv.reader(f)]
        pairs = [(x,y) for x,y in pairs if os.path.exists(x)]
        assert len(pairs) == len(samples), "Cannot associate predictions with input filenames"
        output_dir = LOCAL_RESULTS / "predictions" / split
        output_dir.mkdir(parents=True, exist_ok=True)
        for (x_path, y_path), sample in zip(pairs, samples):
            images, labels = processTestIm(sample)
            tiles = net(images.to(device))["out"].argmax(dim=1).cpu().numpy()
            prediction = np.block([[tiles[0],tiles[1]],[tiles[2],tiles[3]]]).astype(np.uint8)
            with rasterio.open(x_path) as src:
                profile = src.profile.copy()
            profile.update(driver="GTiff", count=1, dtype="uint8", nodata=None)
            chip = Path(x_path).stem.removesuffix("_S1Hand").removesuffix("_S1")
            with rasterio.open(output_dir / f"{chip}_S1Hand_prediction.tif", "w", **profile) as dst:
                dst.write(prediction, 1)
        print("Saved", len(samples), "predictions to", output_dir)


In [ ]:
# Readable sample plots for v1.1; the batch report also shows v1.0 SAR and masks.
if DATASET_VERSION == "v1.1":
    from sen1floods11_visualization import plot_predictions
    import matplotlib.pyplot as plt
    for split in ("non_bol", "bol"):
        fig, axes, chip_ids = plot_predictions(
            predictions_dir=LOCAL_RESULTS / "predictions" / split,
            hand_dir=DATA_ROOT, n=3, seed=42, show_errors=True,
            save_path=LOCAL_RESULTS / f"preview_{split}.png")
        plt.show()
